# Modèles de langue n-grammes — la probabilité d'une phrase

Un n-gramme prédit chaque mot à partir de ses `n-1` prédécesseurs. Ce carnet estime les modèles NLTK **MLE**, **Laplace** et **Kneser-Ney interpolé**, puis mesure leur perplexité sur un test tenu à l'écart. Les probabilités sont agrégées en log-espace : un événement de probabilité nulle rend la perplexité infinie.

**Version jumelle** : [Python](02_NGrammes_Modeles_De_Langue.ipynb).  
**Objectifs** : estimer des n-grammes, gérer les frontières et les inconnus, interpréter la perplexité et générer des phrases.  
**Prérequis** : C#, probabilités et NLTK avec `punkt_tab` français installé. **Durée** : 75 minutes.
**Interopérabilité** : installer CPython avec NLTK et sa ressource `punkt_tab` française. Avant Jupyter, définir `PYTHONNET_PYDLL` vers la bibliothèque partagée et, si nécessaire, `PYTHONHOME`, `PYTHONPATH` et `NLTK_DATA`. Python.NET appelle les API NLTK pour la tokenisation et les modèles ; l’agrégation et les contrôles sont en C#. Aucun téléchargement n’est effectué.


## Comment lire ce notebook

Chaque section entraîne un modèle, mesure les zéros et la perplexité sur le test, puis interprète les sorties. Le train contient 358 tokens NLTK ; le test en contient 70. En bigramme avec frontières, MLE a 43 zéros sur 76 événements, Laplace a une perplexité de 109,021 sans zéro, et Kneser-Ney possède 15 zéros et une perplexité infinie. Ces nombres dépendent du corpus et des conventions de vocabulaire et de padding.


**Navigation** : [<< Précédent — 01. TAL, du mot aux dépendances](01_TAL_Du_Mot_Aux_Dependances.ipynb) | [Index de la série](README.md) | **02 (ce notebook)** | [Suivant — 03. CRF et étiquetage séquentiel >>](03_CRF_Etiquetage_Sequentiel.ipynb)


## 1. Corpus fil rouge -- un seul texte, deux partitions


Le corpus synthétique d'extraits de fables est séparé en entraînement et test : **299 et 64 mots séparés par des espaces**. La tokenisation NLTK, qui sépare aussi la ponctuation, donnera d'autres comptes. Ce petit banc sert à examiner les effets des comptes nuls, non à classer les modèles en général.

In [1]:
#r "nuget: pythonnet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text;
using Microsoft.DotNet.Interactive.Formatting;
using Python.Runtime;
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((v,w)=>w.Write(v.Content),"text/html");
string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);
string fablesTrain="Maitre Corbeau, sur un arbre perche, tenait en son bec un fromage.\nMaitre Renard, par l'odeur alleche, lui tint a peu pres ce langage.\nHe bonjour, Monsieur du Corbeau. Que vous etes joli, que vous me semblez beau.\nSans mentir, si votre ramage se rapporte a votre plumage, vous etes le Phenix des hotes de ces bois.\nLe corbeau, honteux et confus, jura, mais un peu tard, qu'on ne l'y prendrait plus.\nLa cigale ayant chante tout l'ete, se trouva fort depourvue quand la bise fut venue.\nPas un seul petit morceau de mouche ou de vermisseau, elle alla crier famine chez la fourmi sa voisine.\nJe vous paierai, lui dit-elle, avant l'aout, foi d'animal, interet et principal.\nLa fourmi n'est pas preteuse, c'est la son moindre defaut.\nQue faisiez-vous au temps chaud, dit-elle a cette emprunteuse.\nNuit et jour a tout venant je chantais, ne vous deplaise.\nVous chantiez, j'en suis fort aise, eh bien dansez maintenant.\nLe lievre et la tortue firent une course, et la tortue, lente mais obstinee,\ngagna contre le lievre trop confiant.\nRien ne sert de courir, il faut partir a point, la patience vaut mieux que la force.\nLe lion, roi des animaux, apprit un jour que les betes de son royaume le craignaient.\nLe rat des villes et le rat des champs se rencontrerent et partagerent leur repas.\nLe loup et l'agneau se desalteraient au meme ruisseau, le loup chercha une querelle a l'agneau.\nLe corbeau et le renard se parlerent longtemps, le renard eut le fromage.\nLe chene et le roseau se disputaient la force, le vent les departagea.\nPerrette et le pot au lait, la poule aux oeufs d'or, le coq et le renard.\nLe savetier et le financier, le sage et le fou, le laboureur et ses enfants.";
string fablesTest="Le corbeau et le renard se rencontrerent au bord de la fontaine.\nLa fourmi et la cigale se parlerent un jour de grand hiver.\nLe loup et l'agneau se croiserent au meme ruisseau.\nLe maitre et son ecolier firent une longue promenade.\nLe roi des animaux convoqua ses sujets au grand complet.\nLa tortue et le lievre reprirent la course au point de depart.";
int WordsBySpace(string s)=>s.Split((char[])null,StringSplitOptions.RemoveEmptyEntries).Length;
Console.WriteLine($".NET {Environment.Version} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version}");
Console.WriteLine($"Train : {WordsBySpace(fablesTrain)} mots-espace ; test : {WordsBySpace(fablesTest)} mots-espace");

Installed Packages pythonnet, 3.0.5

.NET 10.0.14 | Python.NET 3.0.5.0


Train : 299 mots-espace ; test : 64 mots-espace


### Lecture du résultat

Le test est conservé à part ; les comptes ci-dessus sont des mots séparés par des espaces et ne sont pas encore les tokens NLTK. La casse est conservée. Les statistiques et perplexités du carnet concernent uniquement ces deux partitions.

## 2. Tokenisation en mots et padding


La casse et la ponctuation sont conservées. Pour un ordre `n`, le pipeline ajoute `n-1` symboles de début et de fin à chaque phrase ; les évaluations utilisent explicitement le même padding. La ressource `punkt_tab` française doit être préinstallée et accessible à NLTK (par exemple via `NLTK_DATA`) ; aucun téléchargement n'est effectué dans ce carnet.

In [2]:
if(!PythonEngine.IsInitialized) {
    var dll=Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    if(string.IsNullOrWhiteSpace(dll)) throw new InvalidOperationException("Définir PYTHONNET_PYDLL avant Jupyter.");
    Runtime.PythonDLL=dll; PythonEngine.Initialize(); PythonEngine.BeginAllowThreads();
}
PyObject nltk=null, prep=null, lm=null; string[][] train=null,test=null;
using(Py.GIL()) {
    nltk=Py.Import("nltk"); prep=Py.Import("nltk.lm.preprocessing"); lm=Py.Import("nltk.lm");
    dynamic nl=nltk; nl.data.find("tokenizers/punkt_tab/french/");
    string[][] Tokeniser(string text) {
        using PyObject phrases=nl.sent_tokenize(text,language:"french"); var rows=phrases.As<string[]>();
        return rows.Select(s=>{using PyObject words=nl.word_tokenize(s,language:"french");return words.As<string[]>();}).ToArray();
    }
    train=Tokeniser(fablesTrain);test=Tokeniser(fablesTest);
    Console.WriteLine($"NLTK {nl.__version__} : train {train.Length} phrases / {train.Sum(x=>x.Length)} tokens ; test {test.Length} phrases / {test.Sum(x=>x.Length)} tokens");
}
for(int n=1;n<=3;n++) {var padded=Enumerable.Repeat("<s>",n-1).Concat(train[0]).Concat(Enumerable.Repeat("</s>",n-1)).ToArray(); Console.WriteLine($"n={n} : début du padding {string.Join(" ",padded.Take(8))}");}

NLTK 3.10.4 : train 22 phrases / 358 tokens ; test 6 phrases / 70 tokens


n=1 : début du padding Maitre Corbeau , sur un arbre perche ,


n=2 : début du padding <s> Maitre Corbeau , sur un arbre perche


n=3 : début du padding <s> <s> Maitre Corbeau , sur un arbre


<null>

### Lecture du résultat

NLTK compte **22 phrases / 358 tokens** au train et **6 / 70** au test, ponctuation incluse. Ces comptes diffèrent des 299/64 mots-espace. L'aperçu montre les frontières ajoutées selon l'ordre ; elles restent distinctes de `<UNK>` dans le vocabulaire du modèle.

La frontière de début a un rôle de contexte : `P(w | <s>)` est la probabilité que le premier token soit `w`. La frontière de fin permet aussi d'apprendre où une phrase se termine.


## 3. Unigramme MLE et perplexite


La fréquence d'un mot sur le train est son estimation MLE. Nous comparons les comptes à la vraie API `nltk.lm.MLE` ; le vocabulaire issu du pipeline est reconstruit pour chaque modèle, afin de conserver les frontières et de ne pas épuiser les générateurs.

In [3]:
PyList ToPySentences(string[][] sentences) {var result=new PyList();foreach(var sent in sentences){using var row=new PyList();foreach(string word in sent){using var item=new PyString(word);row.Append(item);}result.Append(row);}return result;}
PyList ToPyStrings(IEnumerable<string> values) {var result=new PyList();foreach(string v in values){using var item=new PyString(v);result.Append(item);}return result;}
// À appeler sous Py.GIL() ; chaque modèle reçoit son propre pipeline.
PyObject Entrainer(string kind,int n,string[][] phrases) {
    using var sentences=ToPySentences(phrases);dynamic p=prep;
    using PyObject pair=p.padded_everygram_pipeline(n,sentences);using var data=pair.GetItem(0);using var vocabulary=pair.GetItem(1);
    using var type=lm.GetAttr(kind);using var order=new PyInt(n);var result=type.Invoke(new PyObject[]{order});dynamic m=result;m.fit(data,vocabulary);return result;
}
double Score(PyObject modele,string word,IEnumerable<string> contexte) {using var ctx=ToPyStrings(contexte);dynamic m=modele;return (double)m.score(word,ctx);}
var countsUni=train.SelectMany(x=>x).GroupBy(x=>x).ToDictionary(g=>g.Key,g=>g.Count());int totalMain=countsUni.Values.Sum();
PyObject lmUni;
using(Py.GIL()) {lmUni=Entrainer("MLE",1,train);dynamic m=lmUni;int count=(int)m.counts.unigrams.N();double expected=(double)countsUni.GetValueOrDefault("corbeau")/totalMain;double actual=Score(lmUni,"corbeau",Array.Empty<string>());
 if(count!=totalMain || Math.Abs(expected-actual)>1e-12)throw new Exception("Oracle unigramme incohérent");
 Console.WriteLine($"Oracle : {totalMain} tokens, {countsUni.Count} types ; P(corbeau) main={expected:F6}, NLTK={actual:F6}");
}

Oracle : 358 tokens, 197 types ; P(corbeau) main=0.005587, NLTK=0.005587


Les comptes MLE sont vérifiés. Calculons maintenant la perplexité et définissons les cas zéro et vide.

In [4]:
(double? Pp,int Zeros,int N) Perplexite(IEnumerable<double> probabilities) {
    var ps=probabilities.ToArray();if(ps.Any(p=>!double.IsFinite(p)||p<0||p>1))throw new ArgumentException("Probabilité invalide");
    if(ps.Length==0)return (null,0,0);int zeros=ps.Count(p=>p==0);if(zeros>0)return (double.PositiveInfinity,zeros,ps.Length);
    double entropy=-ps.Sum(Math.Log)/ps.Length;return (entropy>Math.Log(double.MaxValue)?double.PositiveInfinity:Math.Exp(entropy),0,ps.Length);
}
var oraclePp=Perplexite(test.SelectMany(s=>s).Select(w=>(double)countsUni.GetValueOrDefault(w)/totalMain));
if(!double.IsPositiveInfinity(Perplexite(new[]{0.5,0.0}).Pp.Value) || Perplexite(Array.Empty<double>()).Pp!=null || !double.IsPositiveInfinity(Perplexite(new[]{double.Epsilon}).Pp.Value))throw new Exception("Contrat perplexité incorrect");
bool invalid=false;try{Perplexite(new[]{double.NaN});}catch(ArgumentException){invalid=true;}if(!invalid)throw new Exception("NaN accepté");
Console.WriteLine($"MLE unigramme : PP={oraclePp.Pp}, zéros={oraclePp.Zeros}/{oraclePp.N} ; vide = NA ; probabilités invalides refusées");

MLE unigramme : PP=Infinity, zéros=15/70 ; vide = NA ; probabilités invalides refusées


#### Que mesure la perplexité ?

Elle est l'exponentielle de la moyenne des log-pertes sur **tous** les événements de test. Un seul zéro rend cette valeur infinie ; un test vide est non applicable. On ne retire pas les événements impossibles du dénominateur.

Lorsque toutes les probabilités sont positives, la perplexité est la moyenne géométrique de leurs inverses. Elle s'interprète comme un nombre effectif de choix : un modèle uniforme sur `|V|` tokens donne une perplexité de `|V|` sur des événements de ce vocabulaire.


### Lecture du résultat

L'oracle de comptage et NLTK concordent sur `corbeau` (2/358 ≈ 0,005587). Sur les **70** tokens de test, **15** ont une probabilité unigramme MLE nulle ; la perplexité standard est donc infinie. Une moyenne conditionnée aux seuls événements non nuls répondrait à une autre question.

### Exercice 1 — Vocabulaire et OOV

Normalisez en minuscules les tokens alphabétiques, puis retournez l'union des types, les types vus seulement au test et leur proportion parmi les **types** du test. Un test vide donne un taux nul.

In [5]:
// Exercice 1 : union des types et taux OOV par type de test.
(HashSet<string> Union,HashSet<string> Oov,double Rate)? VocabulaireOov(string[][] a,string[][] b) {
    // TODO : minuscules, mots alphabétiques uniquement, test vide → 0.
    return null;
}
var ex1=VocabulaireOov(new[]{new[]{"Chat","CHAT","123","chien"}},new[]{new[]{"chat","Chien","lapin","lapin","123"}});
if(ex1==null)Console.WriteLine("Exercice 1 à compléter");
else {var autre=VocabulaireOov(new[]{new[]{"a"}},new[]{new[]{"b","c"}});var vide=VocabulaireOov(Array.Empty<string[]>(),Array.Empty<string[]>());
 bool ok=autre.HasValue && vide.HasValue && ex1.Value.Union!=null && ex1.Value.Oov!=null && ex1.Value.Union.SetEquals(new[]{"chat","chien","lapin"}) && ex1.Value.Oov.SetEquals(new[]{"lapin"}) && double.IsFinite(ex1.Value.Rate) && Math.Abs(ex1.Value.Rate-1.0/3)<1e-9 && autre.Value.Union.SetEquals(new[]{"a","b","c"}) && autre.Value.Oov.SetEquals(new[]{"b","c"}) && autre.Value.Rate==1 && vide.Value.Union.Count==0 && vide.Value.Oov.Count==0 && vide.Value.Rate==0;
 Console.WriteLine(ok?"Fixtures OOV validées":"Réponse incorrecte");}

Exercice 1 à compléter


## 4. Bigramme MLE -- et l'explosion des zeros


Le bigramme prédit chaque token et la fin de phrase depuis le token précédent. L'évaluation active explicitement le padding à gauche et à droite ; une probabilité nulle rend la perplexité infinie.

Un contexte plus long peut distinguer davantage de suites, mais les combinaisons possibles se multiplient et beaucoup restent absentes d'un petit train : leurs comptes sont clairsemés.


In [6]:
IEnumerable<string[]> Evenements(string[][] phrases,int n) {
    foreach(var sentence in phrases) {var padded=Enumerable.Repeat("<s>",n-1).Concat(sentence).Concat(Enumerable.Repeat("</s>",n-1)).ToArray();
        for(int i=0;i+n<=padded.Length;i++)yield return padded.Skip(i).Take(n).ToArray();}
}
((double? Pp,int Zeros,int N) Stats,List<string[]> ZeroEvents) Evaluer(PyObject modele,string[][] phrases,int n) {
    var events=Evenements(phrases,n).ToArray();var probs=events.Select(ng=>Score(modele,ng[^1],ng.Take(n-1))).ToArray();return (Perplexite(probs),events.Where((x,i)=>probs[i]==0).ToList());
}
PyObject lmBi; (double? Pp,int Zeros,int N) biStats;
using(Py.GIL()) {lmBi=Entrainer("MLE",2,train);var (stats,zeros)=Evaluer(lmBi,test,2);biStats=stats;
 Console.WriteLine($"Bigrammes test : N={stats.N}, zéros={stats.Zeros}, PP={stats.Pp}");foreach(var ng in zeros.Take(6)) Console.WriteLine($"P({ng[^1]} | {ng[0]}) = 0");
 dynamic m=lmBi;using PyObject begin=m.vocab.lookup("<s>");using PyObject end=m.vocab.lookup("</s>");using PyObject unk=m.vocab.lookup("jamais_vu");
 if(begin.As<string>()!="<s>" || end.As<string>()!="</s>" || unk.As<string>()!="<UNK>")throw new Exception("Vocabulaire incorrect");
 Console.WriteLine($"Vocabulaire bigramme : {(int)m.vocab.__len__()}, début={begin}, fin={end}, inconnu={unk}");
}

Bigrammes test : N=76, zéros=43, PP=Infinity


P(au | rencontrerent) = 0


P(bord | au) = 0


P(de | bord) = 0


P(la | de) = 0


P(fontaine | la) = 0


P(. | fontaine) = 0


Vocabulaire bigramme : 200, début=<s>, fin=</s>, inconnu=<UNK>


### Lecture du résultat

Avec frontières, **43 des 76 bigrammes** du test ont une probabilité MLE nulle. Les six exemples affichés localisent certains échecs ; un seul zéro suffit à rendre la perplexité infinie. Le vocabulaire bigramme comporte 200 entrées, frontières et `<UNK>` comprises.

## 5. Bigramme Laplace add-1 -- tout devient possible


Laplace add-1 attribue à chaque mot la probabilité `(c(h,w)+1)/(c(h)+|V|)`. Pour un contexte connu, la masse d'un événement absent dépend donc aussi de `c(h)` ; elle n'est pas toujours `1/|V|`. Le vocabulaire comprend les frontières du pipeline et `<UNK>`.

Pour un contexte donné, add-1 attribue une masse aux mots non observés après ce contexte ; cette masse est prise sur celle que le MLE accordait aux mots observés. Le compromis dépend des comptes et du vocabulaire.


In [7]:
PyObject lmLap; (double? Pp,int Zeros,int N) lapStats;
using(Py.GIL()) {lmLap=Entrainer("Laplace",2,train);var (stats,_)=Evaluer(lmLap,test,2);lapStats=stats;dynamic l=lmLap;dynamic m=lmBi;
 if(stats.Zeros!=0 || (int)l.counts.unigrams.N()!=(int)m.counts.unigrams.N())throw new Exception("Laplace ou pipeline incorrect");
 Console.WriteLine($"Laplace : N={stats.N}, zéros={stats.Zeros}, PP={stats.Pp:F3} ; MLE : N={biStats.N}, zéros={biStats.Zeros}, PP={biStats.Pp}");
 // Oracle sur deux phrases : même vocabulaire et mêmes frontières, sans réutiliser de générateur.
 var tiny=new[]{new[]{"a","b"},new[]{"a","c"}};using var tinyM=Entrainer("MLE",2,tiny);using var tinyL=Entrainer("Laplace",2,tiny);using var tinyK=Entrainer("KneserNeyInterpolated",2,tiny);
 double mle=Score(tinyM,"b",new[]{"a"}),lap=Score(tinyL,"b",new[]{"a"});
 if(Math.Abs(mle-0.5)>1e-12 || Math.Abs(lap-0.25)>1e-12 || Score(tinyM,"jamais_vu",new[]{"b"})!=0)throw new Exception("Oracle MLE/Laplace incorrect");
 var tinyVocabulary=new[]{"<s>","a","b","</s>","c","<UNK>"};
 foreach(var model in new[]{tinyM,tinyL,tinyK}) if(Math.Abs(tinyVocabulary.Sum(w=>Score(model,w,new[]{"a"}))-1)>1e-12) throw new Exception("Distribution non normalisée");
 Console.WriteLine($"Oracle indépendant : MLE P(b|a)={mle:F3}, Laplace={lap:F3} ; les trois distributions sont normalisées");
}

Laplace : N=76, zéros=0, PP=109.021 ; MLE : N=76, zéros=43, PP=Infinity


Oracle indépendant : MLE P(b|a)=0.500, Laplace=0.250 ; les trois distributions sont normalisées


### Lecture du résultat

Laplace n'a aucun zéro parmi les **76** événements et sa perplexité est **109,021** ; le MLE sur ces mêmes événements vaut `inf`. La formule add-1 distribue une masse positive dans chaque contexte. Ce résultat décrit ce petit corpus et ce traitement des inconnus, pas un classement général.

### Exercice 2 — Lissage add-k

Généralisez la formule add-1 à `k ≥ 0`. Comparez plusieurs valeurs et un mot absent ; un dénominateur nul n'est pas une probabilité définie.

In [8]:
// Exercice 2 : lissage add-k.
double? LissageAddK(Dictionary<(string,string),int> couples,Dictionary<string,int> contextes,string contexte,string mot,double k,int v) {
    // TODO : (c(h,w)+k)/(c(h)+kV).
    return null;
}
var cEx=new Dictionary<(string,string),int>{{("h","a"),2},{("h","b"),1}};var hEx=new Dictionary<string,int>{{"h",4}};
var ex2=LissageAddK(cEx,hEx,"h","a",1,3);
if(ex2==null)Console.WriteLine("Exercice 2 à compléter");
else {var autre=LissageAddK(cEx,hEx,"h","a",0.5,3);var absent=LissageAddK(cEx,hEx,"h","z",1,3);
 bool ok=autre.HasValue && absent.HasValue && double.IsFinite(ex2.Value) && double.IsFinite(autre.Value) && double.IsFinite(absent.Value) && Math.Abs(ex2.Value-3.0/7)<1e-9 && Math.Abs(autre.Value-5.0/11)<1e-9 && Math.Abs(absent.Value-1.0/7)<1e-9;
 Console.WriteLine(ok?"Fixtures add-k validées":"Réponse incorrecte");}

Exercice 2 à compléter


## 6. Bigramme Kneser-Ney interpole -- la continuite des mots rares


Kneser-Ney dit : un mot est rare dans le corpus parce qu'on l'a observe peu, ou parce qu'il apparait dans peu de **contextes** distincts. Le second cas est continu avec la distribution observee. Le lissage est :

```
P_kn(w | h) = max(compte(h, w) - d, 0) / compte(h) + lambda(h) * P_continuation(w)
P_continuation(w) = |{h' : w apparait apres h'}| / |{ (h'', w') avec w' apparaissant apres h'' }|
```

`P_continuation` pondere par la diversite de contextes plutot que par la frequence brute. `nltk.lm.KneserNeyInterpolated` implemente la version interpolee (avec `d`).


In [9]:
PyObject lmKn; (double? Pp,int Zeros,int N) knStats;
using(Py.GIL()) {lmKn=Entrainer("KneserNeyInterpolated",2,train);var (stats,_)=Evaluer(lmKn,test,2);knStats=stats;dynamic k=lmKn;dynamic l=lmLap;
 if((int)k.counts.unigrams.N()!=(int)l.counts.unigrams.N())throw new Exception("Pipeline KN épuisé");
 Console.WriteLine($"Kneser-Ney bigramme : N={stats.N}, zéros={stats.Zeros}, PP={stats.Pp} ; Laplace PP={lapStats.Pp:F3}");
}

Kneser-Ney bigramme : N=76, zéros=15, PP=Infinity ; Laplace PP=109.021


### Lecture du résultat

Kneser-Ney donne **15 zéros sur 76** événements et une perplexité infinie, contre **109,021** pour Laplace sur ces mêmes événements. Les mots hors vocabulaire peuvent rester impossibles : une probabilité nulle n'est ni ignorée ni assimilée à une erreur d'API. La comparaison est limitée à ce corpus.

## 7. Courbe perplexite vs n -- trouver la granularite juste


Les trois familles sont entraînées séparément à chaque ordre. Pour `n=1`, Kneser-Ney n'est pas applicable dans cette API : son estimation de continuation exige des bigrammes. Tous les autres résultats gardent les zéros dans la perplexité. Le nombre d'événements augmente avec le padding, de sorte qu'il ne faut pas classer sans précaution des ordres aux dénominateurs différents.

In [10]:
var results=new Dictionary<(int,string),(double? Pp,int Zeros,int N)>();var models=new Dictionary<(int,string),PyObject>();
using(Py.GIL()) {
 for(int n=1;n<=3;n++) {int expectedCount=-1;
  foreach(var (label,kind) in new[]{("MLE","MLE"),("Laplace","Laplace"),("Kneser-Ney","KneserNeyInterpolated")}) {
   if(n==1 && label=="Kneser-Ney"){results[(n,label)]=(null,0,0);continue;}
   var m=Entrainer(kind,n,train);models[(n,label)]=m;dynamic dm=m;int count=(int)dm.counts.unigrams.N();
   if(count<=0 || (expectedCount>=0 && expectedCount!=count))throw new Exception("Pipeline épuisé");expectedCount=count;
   var (stats,_)=Evaluer(m,test,n);results[(n,label)]=stats;
   Console.WriteLine($"n={n} {label,-12} PP={(stats.Pp.HasValue?stats.Pp.Value.ToString("F3",CultureInfo.InvariantCulture):"NA")} zéros={stats.Zeros} N={stats.N} train={count} vocab={(int)dm.vocab.__len__()}");
  }
  if(n==1)Console.WriteLine("n=1 Kneser-Ney : NA (continuations indisponibles)");
 }
}
var fig=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='760' height='380' viewBox='0 0 760 380'><rect width='760' height='380' fill='white'/><text x='380' y='24' text-anchor='middle' font-size='16'>Perplexité et événements impossibles</text>");
double left=90,top=60,width=570,height=210,maxY=200;
for(int j=0;j<=4;j++){double y=top+height-j*height/4;fig.Append($"<line x1='{F(left)}' y1='{F(y)}' x2='{F(left+width)}' y2='{F(y)}' stroke='#ddd'/><text x='82' y='{F(y+4)}' text-anchor='end' font-size='12'>{F(maxY*j/4)}</text>");}
fig.Append("<path d='M90 60 V270 H660' fill='none' stroke='black'/><text x='90' y='48' font-size='12'>perplexité (finie)</text><text x='375' y='315' text-anchor='middle' font-size='12'>ordre n</text>");
string[] colors={"#555","#2465a8","#b14242"};string[] labels={"MLE","Laplace","Kneser-Ney"};
for(int i=0;i<3;i++){string label=labels[i];var points=new List<string>();for(int n=1;n<=3;n++){double x=left+(n-1)*width/2;var value=results[(n,label)].Pp;if(i==0)fig.Append($"<text x='{F(x)}' y='290' text-anchor='middle' font-size='12'>{n}</text>");
 if(value.HasValue && double.IsFinite(value.Value)){double y=top+height-value.Value/maxY*height;points.Add($"{F(x)},{F(y)}");fig.Append($"<circle cx='{F(x)}' cy='{F(y)}' r='4' fill='{colors[i]}'/>");}
 else {string text=value.HasValue?"inf":"NA";fig.Append($"<text x='{F(x)}' y='{F(77+i*16)}' text-anchor='middle' fill='{colors[i]}' font-size='11'>{label}: {text}</text>");}}
 if(points.Count>0)fig.Append($"<polyline points='{string.Join(" ",points)}' fill='none' stroke='{colors[i]}' stroke-width='2'/>");
 double lx=120+i*180;fig.Append($"<line x1='{F(lx)}' y1='345' x2='{F(lx+20)}' y2='345' stroke='{colors[i]}' stroke-width='2'/><text x='{F(lx+25)}' y='349' font-size='12'>{label}</text>");}
fig.Append("</svg>");display(new PlotSvg(fig.ToString()));

n=1 MLE          PP=Infinity zéros=15 N=70 train=358 vocab=198


n=1 Laplace      PP=141.696 zéros=0 N=70 train=358 vocab=198


n=1 Kneser-Ney : NA (continuations indisponibles)


n=2 MLE          PP=Infinity zéros=43 N=76 train=402 vocab=200


n=2 Laplace      PP=109.021 zéros=0 N=76 train=402 vocab=200


n=2 Kneser-Ney   PP=Infinity zéros=15 N=76 train=402 vocab=200


n=3 MLE          PP=Infinity zéros=58 N=82 train=446 vocab=200


n=3 Laplace      PP=126.881 zéros=0 N=82 train=446 vocab=200


n=3 Kneser-Ney   PP=Infinity zéros=15 N=82 train=446 vocab=200


Perplexité et événements impossibles 0 50 100 150 200 perplexité (finie) ordre n 1 MLE: inf 2 MLE: inf 3 MLE: inf MLE Laplace Kneser-Ney: NA Kneser-Ney: inf Kneser-Ney: inf Kneser-Ney

### Lecture du résultat

Le MLE a **15/70**, **43/76** et **58/82** zéros aux ordres 1, 2 et 3 : ses trois perplexités sont donc infinies. Laplace donne **141,696**, **109,021** et **126,881** sans zéro. Kneser-Ney est `NA` à l'ordre 1 et possède 15 zéros aux ordres 2 et 3. La figure annote `inf` et `NA` sans les transformer en valeurs nulles. Le nombre d'événements varie avec le padding, ce qui limite une comparaison brute entre ordres.

### Exercice 3 — Courbe de perplexité jusqu'à n=4

Entraînez séparément Laplace et Kneser-Ney aux ordres 1 à 4 et tracez leurs perplexités. Retournez les valeurs par famille et ordre ; KN ordre 1 et test vide sont `None`. Les fixtures comparent deux petits corpus, sans choisir d'hyperparamètre sur le test.

In [11]:
// Exercice 3 : courbes Laplace et Kneser-Ney jusqu'à n=4.
Dictionary<string,Dictionary<int,double?>> CourbePerplexite(string[][] a,string[][] b,int maxN=4) {
    // TODO : entraîner des pipelines indépendants ; KN ordre 1 et test vide → null.
    return null;
}
var ex3=CourbePerplexite(new[]{new[]{"a","b"},new[]{"a","c"}},new[]{new[]{"a","b"}},4);
if(ex3==null)Console.WriteLine("Exercice 3 à compléter");
else {
 var autre=CourbePerplexite(new[]{new[]{"x","x"},new[]{"x","y"}},new[]{new[]{"x","y"}},2);
 var vide=CourbePerplexite(new[]{new[]{"a","b"}},Array.Empty<string[]>(),2);
 var expectedLap=new[]{3.265986323711,3.342198623310,3.380969111511,3.404446951435};var expectedKn=new[]{0.0,1.330968217997,1.194970321600,1.149141625891};
 bool ok=ex3.Count==2 && ex3.ContainsKey("Laplace") && ex3.ContainsKey("Kneser-Ney") && ex3["Laplace"].Count==4 && ex3["Kneser-Ney"].Count==4 && Enumerable.Range(1,4).All(n=>ex3["Laplace"].TryGetValue(n,out var x) && x.HasValue && double.IsFinite(x.Value) && Math.Abs(x.Value-expectedLap[n-1])<1e-8) && ex3["Kneser-Ney"].ContainsKey(1) && !ex3["Kneser-Ney"][1].HasValue && Enumerable.Range(2,3).All(n=>ex3["Kneser-Ney"].TryGetValue(n,out var x) && x.HasValue && double.IsFinite(x.Value) && Math.Abs(x.Value-expectedKn[n-1])<1e-8) && autre!=null && autre.Count==2 && autre.ContainsKey("Laplace") && autre.ContainsKey("Kneser-Ney") && autre["Laplace"].Count==2 && autre["Laplace"].ContainsKey(2) && autre["Kneser-Ney"].Count==2 &&
 vide!=null && vide.Count==2 && new[]{"Laplace","Kneser-Ney"}.All(k=>vide.ContainsKey(k) && vide[k].Count==2 && vide[k].ContainsKey(1) && vide[k].ContainsKey(2) && !vide[k][1].HasValue && !vide[k][2].HasValue) && autre["Laplace"].TryGetValue(1,out var a) && a.HasValue && Math.Abs(a.Value-2.474873734153)<1e-8 && autre["Kneser-Ney"].TryGetValue(1,out var b) && !b.HasValue && autre["Kneser-Ney"].TryGetValue(2,out var c) && c.HasValue && Math.Abs(c.Value-1.507706514275)<1e-8;
 Console.WriteLine(ok?"Fixtures courbe validées":"Réponse incorrecte");
}

Exercice 3 à compléter


## 8. Génération d'échantillons — ce que chaque ordre sait

La perplexité compare des modèles sur un corpus de test ; la **génération**
les montre à l'œuvre. En échantillonnant mot à mot depuis chaque MLE
(unigramme, bigramme, trigramme) avec une graine fixe, on lit directement ce
que chaque ordre capture — et ce qui lui échappe. Le critère d'acceptation de
la série (#16235) demande que cette génération soit **visible en sortie** :
chaque modèle produit ci-dessous ses propres échantillons.

In [12]:
using(Py.GIL()) {
 foreach(var (n,seed) in new[]{(1,42),(2,7),(3,11)}) {dynamic m=models[(n,"MLE")];PyObject generated;
  if(n==1)generated=m.generate(12,random_seed:seed);
  else {using var ctx=ToPyStrings(Enumerable.Repeat("<s>",n-1));generated=m.generate(12,text_seed:ctx,random_seed:seed);}
  using(generated) {var words=generated.As<string[]>();int end=Array.IndexOf(words,"</s>");if(end>=0)words=words.Take(end).ToArray();Console.WriteLine($"n={n}, graine={seed} : {(words.Length==0?"(fin immédiate)":string.Join(" ",words))}");}
 }
}

n=1, graine=42 : le , au Phenix ne les ses , disputaient , Perrette faut


n=2, graine=7 : Le corbeau et jour que les betes de ces bois .


n=3, graine=11 : Le lion , roi des animaux , apprit un jour que les


**Lecture des échantillons.** Avec les graines fixées, l'unigramme juxtapose des mots du train ; le bigramme commence par « Le corbeau et jour » et le trigramme par « Le lion, roi des animaux ». Ces exemples illustrent la portée locale du contexte, pas une grammaire globale ni une mesure de qualité générale.

L'unigramme ignore les voisins ; le bigramme tient compte du dernier token et le trigramme des deux derniers. Un contexte plus long peut prolonger des fragments locaux, sans assurer les accords à distance ou la structure globale de la phrase.


## 9. Exercices — récapitulatif

Les exercices répartis portent sur les [types OOV](#exercice-1--vocabulaire-et-oov), le [lissage add-k](#exercice-2--lissage-add-k) et la [courbe jusqu'à n=4](#exercice-3--courbe-de-perplexité-jusquà-n4). Chaque stub reste à compléter ; ses fixtures comprennent des cas contrastés et des entrées vides.

## À retenir

- NLTK traite ici **358 tokens train** et **70 test**. Les frontières restent distinctes de `<UNK>` et chaque modèle reçoit son propre pipeline.
- Un zéro rend la perplexité infinie : MLE produit **15/70** zéros en unigramme et **43/76** en bigramme. Un test vide est non applicable.
- À ordre 2, Laplace obtient **109,021** sans zéro ; Kneser-Ney a **15/76** zéros et une perplexité infinie. L'ordre 1 de Kneser-Ney est `NA` dans cette API.
- Le nombre d'événements varie avec le padding ; les chiffres de ce petit test ne fondent pas un classement général. Les générations illustrent le contexte local.

## Vérifiez votre compréhension

1. Pourquoi une seule probabilité nulle rend-elle la perplexité infinie ?
2. Pourquoi faut-il reconstruire le pipeline avant chaque entraînement ?
3. Pourquoi l'ordre 1 de Kneser-Ney est-il noté `NA` ?
4. Pourquoi les nombres d'événements changent-ils avec `n` ?

<details><summary>Réponses</summary>

1. La log-perte de cet événement est infinie ; il reste dans le dénominateur.
2. Les générateurs sont consommés par `fit`.
3. L'implémentation de continuation requiert des bigrammes.
4. Le padding ajoute `n-1` frontières de chaque côté de chaque phrase.

</details>

## Pour aller plus loin

**Carnet suivant** : [03_CRF_Etiquetage_Sequentiel.ipynb](03_CRF_Etiquetage_Sequentiel.ipynb) — l'étiquetage séquentiel, où la structure de la sortie remplace l'indépendance des tokens.

- **Lectures** : Jurafsky & Martin, *Speech and Language Processing*, ch. « N-gram Language Models » (lissage et perplexité, jusqu'au Kneser-Ney interpolé) ; Manning & Schütze, *Foundations of Statistical Natural Language Processing*, ch. 6.
- **Dans le dépôt** : [01_TAL_Du_Mot_Aux_Dependances.ipynb](01_TAL_Du_Mot_Aux_Dependances.ipynb) (lemmatiser avant d'estimer : moins de types, moins de zéros) ; la série `RAG-et-Memoire-Semantique`, pour prolonger l'étude des modèles de recherche.
